<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 3 · Python Infrastructure for Asset Management Research

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals

This notebook turns Chapter 3 concepts into runnable patterns:

- verify your environment and library versions,
- load/clean the <code>pyaiam_eod.csv</code> dataset,
- organize data into long/wide formats with reproducible helper functions,
- experiment with plotting defaults, and
- sketch a light project skeleton with reusable utilities.

Everything runs locally or in Colab without extra infrastructure.

### Getting Help While Coding
- **Chapter 3** itself walks through environment setup and workflow tips.
- **Appendix B** is your quick lookup for NumPy/pandas idioms used here.
- **Appendix F** covers practical tooling (logging, configs) referenced later.
Revisit them whenever new syntax pops up.

In [ ]:
import platform
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"


## 1. Environment Snapshot
Capturing package versions inside notebooks makes it easy to reproduce results later.

In [ ]:
env = pd.Series(
    {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": plt.matplotlib.__version__,
    }
)
env

## 2. Load the Multi-Asset Panel
_Need a reminder on file IO? See Chapter 3 (data ingestion) or Appendix B._

In [ ]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
prices.head()

### 2.1 Data Quality Checks
Forward-filling missing values keeps FX gaps (EURUSD) manageable while flagging any assets with too many NaNs.

In [ ]:
missing = prices.isna().sum()
share_missing = missing.div(len(prices))
pd.DataFrame({"missing": missing, "pct_missing": share_missing})

In [ ]:
prices_clean = prices.ffill()
log_rets = np.log(prices_clean / prices_clean.shift(1)).dropna()
log_rets.head()

## 3. Wide vs. Long Formats
Research notebooks often pivot between time-indexed tables and long-form panels for grouped operations.

In [ ]:
panel_long = (
    prices_clean.reset_index()
    .melt(id_vars="Date", var_name="ticker", value_name="price")
    .sort_values(["Date", "ticker"])
)
panel_long.head()

### 3.1 Resampling Utilities
Weekly summaries support slide decks and production monitors alike.

In [ ]:
def resample_prices(price_frame: pd.DataFrame, freq: str = "W") -> pd.DataFrame:
    return price_frame.resample(freq).last()

weekly = resample_prices(prices_clean)
weekly.head()

## 4. Rolling Feature Engineering
Appendix B covers the pandas rolling API; here we wrap a reusable function for signal creation.

In [ ]:
def rolling_features(log_returns: pd.DataFrame, window: int = 21) -> pd.DataFrame:
    vol = log_returns.rolling(window).std() * np.sqrt(252)
    mom = log_returns.rolling(window).sum()
    out = pd.concat({"vol": vol, "mom": mom}, axis=1)
    return out.dropna()

features = rolling_features(log_rets)
features.head()

### 4.1 Visualization Defaults
Matplotlib handles styling globally; sticking to consistent figure sizes keeps presentations polished.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
(prices_clean[["AAPL", "JPM", "SPY"]] / prices_clean[["AAPL", "JPM",
"SPY"]].iloc[0]).plot(ax=ax)
ax.set_title("Indexed Price Paths (start = 1.0)")
ax.set_ylabel("Index level")
plt.show()

## 5. Lightweight Project Skeleton
Even in notebooks, modular helpers reduce copy/paste.

In [ ]:
PROJECT_ROOT = Path("../")
STRUCTURE = {
    "data": ["raw", "intermediate", "features"],
    "notebooks": [],
    "reports": ["figures", "tables"],
}

def print_structure(root: Path, structure: dict) -> None:
    for folder, subfolders in structure.items():
        print(f"{root / folder}")
        for sub in subfolders:
            print(f"    {root / folder / sub}")

print_structure(PROJECT_ROOT, STRUCTURE)

### 5.1 Configuration Helper
Store common paths/settings in a dictionary to pass into scripts or pipelines.

In [ ]:
CONFIG = {
    "data_file": DATA_PATH,
    "feature_window": 21,
    "plots_dir": PROJECT_ROOT / "reports" / "figures",
}
CONFIG

## 6. Exercises
### Exercise 1 – Weekly Return Pipeline
Create <code>weekly_returns(prices)</code> that returns compounded weekly returns for any subset of tickers.
<details><summary>Hint</summary>
Combine <code>resample('W').last()</code> with <code>pct_change()</code> and drop NaNs.
</details>

### Exercise 2 – Data Validation Guardrail
Write a function that raises an exception if any asset has more than 5&#37; missing data prior to forward-filling.
<details><summary>Hint</summary>
Reuse the <code>share_missing</code> logic above and compare against 0.05.
</details>

### Exercise 3 – Notebook Bootstrap Cell
Draft a Colab-friendly cell that clones the repo (if needed), installs dependencies, downloads <code>pyaiam_eod.csv</code>, and mounts Google Drive.
<details><summary>Hint</summary>
Stack shell commands via <code>%%bash</code> or Python's <code>subprocess</code> plus <code>gdown</code>/<code>curl</code> for data.
</details>


## 7. Takeaways for Chapter 3
- Capturing environment info and consistent paths makes notebooks reproducible.
- Helper functions for resampling/rolling stats reduce friction in later chapters.
- Lightweight project skeletons keep experiments organized even before formal packaging.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">